# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [1]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 17:26:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [2]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail.parquet")
)

transactions = (
    sdf
    .select("InvoiceNo", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["InvoiceNo"], (row["StockCode"], int(row["Quantity"]))))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

[('536365',
  [('85123A', 6),
   ('71053', 6),
   ('84406B', 8),
   ('84029G', 6),
   ('84029E', 6),
   ('22752', 2),
   ('21730', 6)]),
 ('536366', [('22633', 6), ('22632', 6)]),
 ('536367',
  [('84879', 32),
   ('22745', 6),
   ('22748', 6),
   ('22749', 8),
   ('22310', 6),
   ('84969', 6),
   ('22623', 3),
   ('22622', 2),
   ('21754', 3),
   ('21755', 3),
   ('21777', 4),
   ('48187', 4)]),
 ('536368', [('22960', 6), ('22913', 3), ('22912', 3), ('22914', 3)]),
 ('536369', [('21756', 3)])]

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [3]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


ci sono 45280 coppie


[(('10002', -3), 0),
 (('10002', 1), 1),
 (('10002', 2), 2),
 (('10002', 3), 3),
 (('10002', 4), 4),
 (('10002', 5), 5),
 (('10002', 6), 6),
 (('10002', 8), 7),
 (('10002', 10), 8),
 (('10002', 11), 9),
 (('10002', 12), 10),
 (('10002', 14), 11),
 (('10002', 18), 12),
 (('10002', 24), 13),
 (('10002', 36), 14),
 (('10002', 48), 15),
 (('10002', 60), 16),
 (('10002', 62), 17),
 (('10002', 120), 18),
 (('10002', 180), 19),
 (('10080', 1), 20),
 (('10080', 2), 21),
 (('10080', 3), 22),
 (('10080', 4), 23),
 (('10080', 12), 24),
 (('10080', 22), 25),
 (('10080', 24), 26),
 (('10080', 26), 27),
 (('10080', 48), 28),
 (('10080', 170), 29),
 (('10120', 1), 30),
 (('10120', 2), 31),
 (('10120', 3), 32),
 (('10120', 4), 33),
 (('10120', 5), 34),
 (('10120', 6), 35),
 (('10120', 8), 36),
 (('10120', 10), 37),
 (('10120', 11), 38),
 (('10120', 12), 39),
 (('10120', 20), 40),
 (('10120', 30), 41),
 (('10123C', -18), 42),
 (('10123C', 1), 43),
 (('10123C', 3), 44),
 (('10123G', -38), 45),
 (('10124

In [4]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)
bc_map.value

{('10002', -3): 0,
 ('10002', 1): 1,
 ('10002', 2): 2,
 ('10002', 3): 3,
 ('10002', 4): 4,
 ('10002', 5): 5,
 ('10002', 6): 6,
 ('10002', 8): 7,
 ('10002', 10): 8,
 ('10002', 11): 9,
 ('10002', 12): 10,
 ('10002', 14): 11,
 ('10002', 18): 12,
 ('10002', 24): 13,
 ('10002', 36): 14,
 ('10002', 48): 15,
 ('10002', 60): 16,
 ('10002', 62): 17,
 ('10002', 120): 18,
 ('10002', 180): 19,
 ('10080', 1): 20,
 ('10080', 2): 21,
 ('10080', 3): 22,
 ('10080', 4): 23,
 ('10080', 12): 24,
 ('10080', 22): 25,
 ('10080', 24): 26,
 ('10080', 26): 27,
 ('10080', 48): 28,
 ('10080', 170): 29,
 ('10120', 1): 30,
 ('10120', 2): 31,
 ('10120', 3): 32,
 ('10120', 4): 33,
 ('10120', 5): 34,
 ('10120', 6): 35,
 ('10120', 8): 36,
 ('10120', 10): 37,
 ('10120', 11): 38,
 ('10120', 12): 39,
 ('10120', 20): 40,
 ('10120', 30): 41,
 ('10123C', -18): 42,
 ('10123C', 1): 43,
 ('10123C', 3): 44,
 ('10123G', -38): 45,
 ('10124A', 1): 46,
 ('10124A', 3): 47,
 ('10124A', 4): 48,
 ('10124G', 4): 49,
 ('10124G', 5): 50,
 

In [5]:
# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

25900


[('536365', [42830, 36531, 38846, 38256, 38217, 23013, 9281]),
 ('536366', [21163, 21139]),
 ('536367',
  [40566,
   22907,
   22950,
   22967,
   16057,
   41167,
   20963,
   20948,
   9508,
   9524,
   9631,
   36258]),
 ('536368', [25890, 25160, 25151, 25170]),
 ('536369', [9539])]

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [6]:
epsilon = 50

#supporto(item) = numero di transazioni che contengono lo stesso item

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


2588


[(23013, 121),
 (42830, 560),
 (20963, 95),
 (9508, 269),
 (22950, 138),
 (20948, 121),
 (9524, 201),
 (40566, 85),
 (22907, 138),
 (25890, 451)]

Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [7]:
# creo f_list
f_list = item_counts.sortBy(
    lambda row: (row[1], row[0]),
    ascending=False
)

f_list.take(10)

[(42551, 724),
 (45185, 708),
 (2002, 631),
 (40552, 597),
 (17796, 596),
 (5318, 583),
 (22426, 571),
 (42830, 560),
 (29333, 551),
 (29516, 513)]

In [8]:
# Creo una mappa per ordinare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)

ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


18067
[('536365', [42830, 23013]), ('536367', [9508, 9524, 22950, 22907, 20948, 20963, 40566]), ('536368', [25890]), ('536369', [9539]), ('536370', [18805, 45252, 9311, 19434, 22552])]


In [9]:
# G-List
Q = spark.sparkContext.defaultParallelism # numero di core disponibili tra tutti i worker

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

In [10]:
# Questo è fondamentalmente il mapper del paper
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()
    
    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

# a ogni gid associo le transazioni di cui si deve occupare.

group_shards = group_dependent_transactions.groupByKey()

### Nodi e Alberi

A questo punto abbiamo bisogno degli FP-tree per minare i pattern. 
Ogni worker/reducer costruirà un FP-tree locale a partire dalle transazioni associate a uno specifico gruppo.

Dato che gli item di ogni transazione sono già ordinati secondo la F-list, e dato che le transazioni group-dependent sono raggruppate per `gid`, ogni gruppo può essere minato in modo indipendente dagli altri (come dimostrato nel paper).

Una transazione originale può generare più transazioni parziali, una per ogni gruppo presente nella transazione. Durante lo shuffle, questi prefissi vengono inviati ai reducer corrispondenti ai rispettivi gruppi.

Quindi una stessa transazione originale può contribuire a più FP-tree locali, ma ogni FP-tree locale contiene solo il sotto-database necessario per minare i pattern che terminano negli item del proprio gruppo.

In [11]:

from collections import defaultdict

# Creiamo una struttura di nodi in grado di navigare al parent e ai child.
class FPNode:
    def __init__(self, item=None, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

    def add_child(self, item):
        child = FPNode(item=item, parent=self)
        self.children[item] = child
        return child


# L'inserimento di una transazione può comportare la creazione di nodi figli oppure l'incremento del loro conteggio.

def insert_transaction(root, transaction, header_table, count=1):
    node = root
    for item in transaction:

        if item in node.children:
            child = node.children[item]
            child.count += count
        else:
            child = node.add_child(item)
            child.count = count
            header_table[item].append(child)

        node = child
    
# La header-table serve per avere una navigazione rapida ai nodi che rappresentano lo stesso item, infatti nell'albero possono 
# esserci N nodi che rappresentano lo stesso item, grazie alla header_table possiamo evitare di navigare tutto l'albero 
# ma abbiamo un accesso "diretto"

def build_fp_tree(transactions_iter):
    root = FPNode()
    header_table = defaultdict(list)

    for transaction in transactions_iter:
        insert_transaction(root, transaction, header_table, count=1)

    return root, dict(header_table) 

def is_single_path(node):
    current = node

    while True:
        if len(current.children) == 0:
            return True
        if len(current.children) > 1:
            return False

        current = next(iter(current.children.values()))

        
def print_tree(node, indent=0, max_depth=3):
    if indent >= max_depth:
        return

    for child in node.children.values():
        print("-" * indent + f"{child.item}:{child.count}")
        print_tree(child, indent + 1, max_depth)



def count_nodes(node):
    total = 1
    for child in node.children.values():
        total += count_nodes(child)
    return total



def tree_stats_for_group(row):
    gid, transactions_iter = row

    root, header_table = build_fp_tree(transactions_iter)

    return (
        gid,
        count_nodes(root),
        len(header_table),
        list(root.children.keys())[:10]
    )

In [12]:
# Ora creiamo la nowGroup

gid_to_items_tmp = defaultdict(list)

for item_id, gid in g_list.items():
    gid_to_items_tmp[gid].append(item_id)
# gid -> item_ids
nowGroup = dict(gid_to_items_tmp)

bc_nowGroup = sc.broadcast(nowGroup)

In [13]:
#tree_stats = group_shards.map(tree_stats_for_group)
#gid, nodes_number, header_table_length,childern_length = tree_stats.take(1)[0]
#tree_stats.take(10)
#print("gid:", gid)
#print("item distinti nell'albero:", header_table_length)

In [14]:
# Dato un nodo, risale fino alla root e restituisce il cammino dei parent.
# Esempio se il nodo è d ed il path è a -> b -> c -> d restituisce [a,b,c] 
# Quindi non torna root e nodo corrente

def get_prefix_path(node):
    path = []
    current = node.parent
    while current is not None and current.item is not None:
        path.append(current.item)
        current = current.parent
    path.reverse()
    return path


# Per un item, prende tutti i nodi in header_table[item] e costruisce una lista degli elementi precedenti a lui nel path


def conditional_pattern_base(item, header_table):
    base = []
    for node in header_table.get(item, []):
        path = get_prefix_path(node)
        if path:
            base.append((path, node.count))
    return base


# conta gli item
# fa pruning degli item sotto soglia
# ordina i prefissi
# costruisce un nuovo FP-tree condizionato

def build_conditional_tree(pattern_base, min_support):
    # 1. conta item nella base pesando con node.count
    item_counts = defaultdict(int)
    for path, count in pattern_base:
        for item in path:
            item_counts[item] += count

    # 2. pruning
    frequent_items = {item for item, c in item_counts.items() if c >= min_support}
    if not frequent_items:
        return None, {}

    # 3. costruisco transazioni filtrate e ordinate
    root = FPNode()
    header_table = defaultdict(list)

    for path, count in pattern_base:
        filtered = [item for item in path if item in frequent_items]
        if filtered:
            insert_transaction(root, filtered, header_table, count=count)

    return root, dict(header_table)


# Per ogni item nell’albero:
# crea il pattern prefix + item
# salva il supporto
# costruisce il conditional tree
# richiama ricorsivamente mine_fp_tree

def mine_fp_tree(root, header_table, min_support, suffix=()):
    patterns = []

    items = sorted(
        header_table.keys(),
        key=lambda item: sum(node.count for node in header_table[item])
    )

    for item in items:
        support = sum(node.count for node in header_table[item])

        if support < min_support:
            continue
            
        new_pattern = tuple(sorted((item,) + suffix))
        patterns.append((new_pattern, support))

        base = conditional_pattern_base(item, header_table)
        cond_root, cond_header = build_conditional_tree(base, min_support)

        if cond_header:
            patterns.extend(
                mine_fp_tree(cond_root, cond_header, min_support, suffix=(item,) + suffix)
            )

    return patterns

In [15]:
#TESTING CODE BOX
gid, transactions_iter = group_shards.take(1)[0]
root, header_table = build_fp_tree(transactions_iter)

patterns = mine_fp_tree(root, header_table, min_support=epsilon)
patterns[:20]

[((20179,), 50),
 ((21368,), 50),
 ((5531,), 50),
 ((16153,), 50),
 ((12004,), 50),
 ((21159,), 50),
 ((13860,), 50),
 ((5583,), 50),
 ((16684,), 50),
 ((22757,), 50),
 ((36817,), 50),
 ((25959,), 50),
 ((43384,), 50),
 ((37826,), 50),
 ((18361,), 50),
 ((24381,), 50),
 ((12298,), 50),
 ((41832,), 50),
 ((26433,), 50),
 ((27193,), 50)]

In [16]:
def mine_group(row):
    gid, transactions_iter = row
    root, header_table = build_fp_tree(transactions_iter)
    return mine_fp_tree(root, header_table, min_support=100)

all_patterns = group_shards.flatMap(mine_group)

In [17]:
# all_patterns.take(1000)

# readable_patterns = all_patterns.map(
#     lambda x: f"pattern={x[0]} | support={x[1]}"
# )

# readable_patterns.saveAsTextFile("output_patterns")

In [18]:
invert_conversion_map = {v:k for k, v in conversion_map.items()}
bc_invert_conversion_map = sc.broadcast(invert_conversion_map)

def convert_to_qt(pattern):
    pattern, support = pattern
    return (tuple(bc_invert_conversion_map.value[i] for i in pattern), support)

real_patterns = all_patterns.map(convert_to_qt).persist()

In [20]:
def parent_map_to_dot(parent_map, graph_name="G"):
    """
    Convert a dict {node_tuple: parent_tuple_or_None} into Graphviz DOT.

    Example input:
        {
            ("B", 2): ("A", 1),
            ("C", 3): ("A", 1),
            ("A", 1): None,
        }
    """
    lines = [f'digraph "{graph_name}" {{']

    # Collect all nodes
    nodes = set(parent_map.keys())
    nodes.update(parent for parent in parent_map.values() if parent is not None)

    # Stable internal IDs
    node_ids = {node: f"n{i}" for i, node in enumerate(sorted(nodes, key=repr))}

    # Node declarations with tuple labels
    for node in sorted(nodes, key=repr):
        label = repr(node).replace('"', r"\"")
        lines.append(f'    {node_ids[node]} [label="{label}"];')

    # Edges: parent -> child
    for child, parent in parent_map.items():
        if parent is not None:
            lines.append(f'    {node_ids[parent]} -> {node_ids[child]};')

    lines.append("}")
    return "\n".join(lines)

def find_proportion(pattern1: tuple, pattern2: tuple) -> float | None:
    from math import isclose
    
    alpha = None
    for i1, qt1 in pattern1:
        qt1 = abs(qt1)
        for i2, qt2 in pattern2:
            qt2 = abs(qt2)
            if alpha is None and i1 == i2:
                alpha = qt1 / qt2
            elif i1 == i2 and not isclose(qt1 / qt2, alpha):
                return None
    
    return alpha

def find_proportion_roots(patterns: list[tuple]) -> dict[tuple, tuple]:
    pattern_parent = {pattern: None for pattern in patterns}

    # Group by items contained in pattern
    items_map = {}
    for pattern in patterns:
        pattern_items = tuple(sorted(item for item, _ in pattern))
        if pattern_items not in items_map:
            items_map[pattern_items] = set()
        items_map[pattern_items].add(pattern)
    
    for items in items_map:
        ordered_patterns = []
        patterns = items_map[items]
        for pattern in patterns:
            ordered_patterns.append(tuple(sorted(pattern, reverse=True, key=lambda x: (x[1], x[0]))))
        
        items_map[items] = list(sorted(ordered_patterns, reverse=True))

    for _, patterns in items_map.items():
        for p1 in patterns:
            if pattern_parent[p1] is not None:
                continue
            for p2 in patterns:
                if pattern_parent[p2] is not None:
                    continue
                if sorted(p1) == sorted(p2):
                    continue

                alpha = find_proportion(p1, p2)
                if alpha is None:
                    continue
                if alpha < 1:
                    p2, p1 = p1, p2
                pattern_parent[p1] = p2
    
    with open('forest.dot', "w") as f:
        f.write(parent_map_to_dot(pattern_parent))

    return pattern_parent

def groupby_alpha(patterns: list[tuple[tuple, int]]) -> dict[tuple, tuple]:
    no_support_patterns = [pattern for pattern, _ in patterns]
    return find_proportion_roots(no_support_patterns)


real_patterns.mapPartitions(groupby_alpha, True).collect()

26/05/11 17:31:07 ERROR Executor: Exception in task 0.0 in stage 45.0 (TID 21)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/mattia/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/home/mattia/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mattia/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/core/rdd.py", line 705, in func
    return f(iterator)
           ^^^^^^^^^^^
  File "/tmp/ipykernel_87024/1340796780.py", line 92, in groupby_alpha
  File "/tmp/ipykernel_87024/1340796780.py", line 70, in find_proportion_roots
KeyError: (('DOT', 1), ('22358', 1))

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:645)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1029)
	at org.apache.

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 45.0 failed 1 times, most recent failure: Lost task 0.0 in stage 45.0 (TID 21) (localhost executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/mattia/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/home/mattia/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mattia/miniconda3/envs/pyspark-lab/lib/python3.11/site-packages/pyspark/core/rdd.py", line 705, in func
    return f(iterator)
           ^^^^^^^^^^^
  File "/tmp/ipykernel_87024/1340796780.py", line 92, in groupby_alpha
  File "/tmp/ipykernel_87024/1340796780.py", line 70, in find_proportion_roots
KeyError: (('DOT', 1), ('223